In [31]:
# System Dependencies 
import os
import json
import time
import pandas as pd
from dotenv import load_dotenv

# Google Vertex AI Core Modules
import vertexai
from vertexai import Client
from vertexai import types
from vertexai.preview import reasoning_engines

# Load configurations 
load_dotenv()
PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT", "ragmanageddb-vertexai")
LOCATION = os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1")
GCS_DESTINATION = os.getenv("GOOGLE_CLOUD_STAGING_BUCKET", "gs://travel-planner-staging-ragmanageddb")

print(f"Initializing connection to Project: {PROJECT_ID} inside {LOCATION}")
vertexai.init(project=PROJECT_ID, location=LOCATION)

# Instantiate the principal Rapid Evaluation Client
client = Client(project=PROJECT_ID, location=LOCATION)
print("Vertex AI Programmatic Client Instantiated Successfully ✅")

Initializing connection to Project: ragmanageddb-vertexai inside us-central1
Vertex AI Programmatic Client Instantiated Successfully ✅


In [32]:
def discover_active_reasoning_engines():
    print("=" * 70)
    print(" RUNNING SYSTEM AUTO-DISCOVERY: VERTEX AI REASONING ENGINES ")
    print("=" * 70)
    
    try:
        # Programmatic fetch using the native Vertex preview service list method
        deployed_agents = reasoning_engines.ReasoningEngine.list()
        discovered_registry = []
        
        if not deployed_agents:
            print("⚠️ No deployed Reasoning Engine agents found in this project infrastructure.")
            return []
            
        for agent in deployed_agents:
            # Safely extract the fully-qualified platform resource string
            resource_name = agent.resource_name
            agent_uuid = resource_name.split("/")[-1]
            
            # Extract the user-defined string or fallback dynamically
            display_name = getattr(agent, "display_name", f"agent-{agent_uuid[:6]}")
            
            # Retrieve system timestamps from the underlying gca proto representation
            create_time_str = "N/A"
            if hasattr(agent, "gca_resource") and hasattr(agent.gca_resource, "create_time"):
                create_time_str = str(agent.gca_resource.create_time)
            
            # Map clean key-value pairs matching your target PostgreSQL 'agents' table layout
            agent_metadata = {
                "id": agent_uuid,
                "name": display_name,
                "deployment_type": "vertex_ai",
                "gcp_project": PROJECT_ID,
                "region": LOCATION,
                "endpoint_url": resource_name,
                "status": "healthy", 
                "model_name": "gemini-1.5-pro",
                "created_at": create_time_str,
                "labels": getattr(agent, "labels", {})
            }
            discovered_registry.append(agent_metadata)
            
            # Print format matching structural details required on the Agent Card UI view
            print(f"📍 Found Agent: {agent_metadata['name']}")
            print(f"   ▫️ Resource String : {agent_metadata['endpoint_url']}")
            print(f"   ▫️ Creation Date   : {agent_metadata['created_at']}")
            print(f"   ▫️ Operational Status: {agent_metadata['status'].upper()}\n")
            
        print(f"✅ Discovery complete. Registered {len(discovered_registry)} platform targets.")
        return discovered_registry
        
    except Exception as e:
        print(f"❌ Critical exception encountered during infrastructure polling: {str(e)}")
        return []

# Run active fleet discovery sync routine
agent_fleet = discover_active_reasoning_engines()

 RUNNING SYSTEM AUTO-DISCOVERY: VERTEX AI REASONING ENGINES 
📍 Found Agent: Travel Planner Agent
   ▫️ Resource String : projects/936666675765/locations/us-central1/reasoningEngines/1913161777202331648
   ▫️ Creation Date   : 2026-05-29 05:24:02.169093+00:00
   ▫️ Operational Status: HEALTHY

✅ Discovery complete. Registered 1 platform targets.


In [33]:
class EvaluationDataIngestor:
    """Handles adaptive conversions of text arrays or files into structural evaluation payloads."""
    
    @staticmethod
    def process_manual_ui_prompts(prompt_list: list) -> pd.DataFrame:
        """Converts raw user strings from the manual UI text area into a dataset matrix."""
        # SessionInput triggers tracing pipelines required by the model judges
        session_inputs = types.evals.SessionInput(user_id="agentops_eval_user", state={})
        
        dataframe_output = pd.DataFrame({
            "prompt": prompt_list,
            "session_inputs": [session_inputs] * len(prompt_list)
        })
        return dataframe_output

    @staticmethod
    def process_uploaded_file(file_path: str, format_type: str = "json") -> pd.DataFrame:
        """Parses dragged files passing schema criteria validation checks."""
        session_inputs = types.evals.SessionInput(user_id="agentops_eval_user", state={})
        
        if format_type == "json":
            raw_df = pd.read_json(file_path)
        else:
            raw_df = pd.read_csv(file_path)
            
        if "prompt" not in raw_df.columns and "input" in raw_df.columns:
            raw_df.rename(columns={"input": "prompt"}, inplace=True)
            
        if "session_inputs" not in raw_df.columns:
            raw_df["session_inputs"] = [session_inputs] * len(raw_df)
            
        return raw_df[["prompt", "session_inputs"]]

# Simulation of interactive UI entry: User submits three baseline validation strings
sample_ui_prompts = [
    "Plan a detailed 1 day vacation path through Goa exploring beaches.",
    "Draft a precise travel itinerary sequence for a historical tour of Paris.",
    "Recommend budget dining options across New Delhi for a solo traveler."
]

# Structure payload dataframe
dataset_for_evaluation = EvaluationDataIngestor.process_manual_ui_prompts(sample_ui_prompts)
print(f"Successfully formatted user evaluation matrix dataset. Rows prepared: {len(dataset_for_evaluation)}")

Successfully formatted user evaluation matrix dataset. Rows prepared: 3


In [37]:
# Target selection setup mirroring the metric switches toggled on your platform UI
MAXIMIZED_MANAGED_METRICS = [
    types.RubricMetric.FINAL_RESPONSE_QUALITY,     # Adaptive Evaluation of outcome completeness
    types.RubricMetric.TEXT_QUALITY,                # Structural language properties & grammar fluency
    types.RubricMetric.INSTRUCTION_FOLLOWING,       # Measures constraint checks on prompt directives
    types.RubricMetric.HALLUCINATION,              # Factuality checking against source material
    types.RubricMetric.TOOL_USE_QUALITY,           # Evaluates task handling accuracy across tool steps
]

# Pick the first discovered agent engine resource target for processing execution
if agent_fleet:
    TARGET_AGENT_RESOURCE = agent_fleet[0]["endpoint_url"]
else:
    # Manual fallback initialization format matching local settings parameters
    TARGET_AGENT_RESOURCE = f"projects/{PROJECT_ID}/locations/{LOCATION}/reasoningEngines/placeholder_id"

print(f"Selected Eval Target Instance: {TARGET_AGENT_RESOURCE}")
print(f"Compiled metric pipelines list for testing array evaluation:\n{[m for m in MAXIMIZED_MANAGED_METRICS]}")

# Execute bulk agent invocation step
print("\n🚀 Invoking target engine instances for bulk inference. Running traces...")
inference_dataset_output = client.evals.run_inference(
    agent=TARGET_AGENT_RESOURCE,
    src=dataset_for_evaluation
)

print("Inference generation processing concluded successfully ✅")

# ------------------------------------------------------------
# FIXED MAPPING PATH: ADAPTIVE INTEGER INDEX RESOLUTION
# ------------------------------------------------------------
print("\n🔍 Evaluating Inference Token Integrity...")

# Extract inner data matrix out of the EvaluationDataset wrapper safely
if hasattr(inference_dataset_output, "dataset"):
    inference_df = inference_dataset_output.dataset
elif isinstance(inference_dataset_output, pd.DataFrame):
    inference_df = inference_dataset_output
else:
    inference_df = pd.DataFrame(inference_dataset_output)

# Clear string column match logic fallbacks
if "response" in inference_df.columns:
    response_col_key = "response"
elif "output" in inference_df.columns:
    response_col_key = "output"
# Dynamic Integer Fallback: Target Column position 1 if string headers are missing
elif 1 in inference_df.columns:
    response_col_key = 1
else:
    response_col_key = None

if response_col_key is None:
    print("❌ ERROR: Could not find a recognizable output column or integer index position in your inference dataset.")
    print(f"Detected columns are: {list(inference_df.columns)}")
else:
    print(f"   ▫️ Successfully targeted response column vector identifier: [{response_col_key}]")
    
    # Isolate missing or empty values defensively to verify agent generation success
    null_or_empty_responses = inference_df[
        inference_df[response_col_key].isna() | 
        (inference_df[response_col_key].astype(str).str.strip() == "") |
        (inference_df[response_col_key].astype(str).str.strip() == "N/A")
    ]

    if len(null_or_empty_responses) > 0:
        print(f"❌ CRITICAL EXCEPTION: Found {len(null_or_empty_responses)} empty generations from your agent.")
        print("Stopping execution loop. Deployed agent generated empty outputs. Check your backend log streams.")
    else:
        print(f"Inference payload verified clean! All {len(inference_df)} test cases populated successfully. 👍")

Selected Eval Target Instance: projects/936666675765/locations/us-central1/reasoningEngines/1913161777202331648
Compiled metric pipelines list for testing array evaluation:
[<vertexai._genai._evals_metric_loaders.LazyLoadedPrebuiltMetric object at 0x000001B0FE2A8280>, <vertexai._genai._evals_metric_loaders.LazyLoadedPrebuiltMetric object at 0x000001B0FE2AA900>, <vertexai._genai._evals_metric_loaders.LazyLoadedPrebuiltMetric object at 0x000001B0FE2A9240>, <vertexai._genai._evals_metric_loaders.LazyLoadedPrebuiltMetric object at 0x000001B0FE2AA6D0>, <vertexai._genai._evals_metric_loaders.LazyLoadedPrebuiltMetric object at 0x000001B0FE2AA4A0>]

🚀 Invoking target engine instances for bulk inference. Running traces...


Agent Run: 100%|██████████| 3/3 [03:16<00:00, 65.53s/it] 

Inference generation processing concluded successfully ✅

🔍 Evaluating Inference Token Integrity...
   ▫️ Successfully targeted response column vector identifier: [1]
❌ CRITICAL EXCEPTION: Found 3 empty generations from your agent.
Stopping execution loop. Deployed agent generated empty outputs. Check your backend log streams.


In [38]:
print("Registering evaluation job with Vertex AI judge cluster...")

try:
    eval_run_handle = client.evals.create_evaluation_run(
        dataset=inference_dataset_output,
        agent=TARGET_AGENT_RESOURCE,
        metrics=MAXIMIZED_MANAGED_METRICS,
        dest=GCS_DESTINATION
    )
    
    print(f"\nJob Token Registered Successfully: {eval_run_handle.name}")
    print(f"Initial State: {eval_run_handle.state}\n")
    
    # Non-blocking async loop simulation to mirror platform task monitoring routines
    while eval_run_handle.state not in {"SUCCEEDED", "FAILED", "CANCELLED"}:
        print(f"⏰ Polling execution status: [Current State: {eval_run_handle.state}] — sleeping 15s...")
        time.sleep(15)
        eval_run_handle = client.evals.get_evaluation_run(name=eval_run_handle.name)
        
    print(f"\n🏁 Lifecycle execution concluded! Terminal State: {eval_run_handle.state}")
    
except Exception as run_error:
    print(f"❌ Fatal evaluation pipeline execution failure encountered: {str(run_error)}")

Registering evaluation job with Vertex AI judge cluster...

Job Token Registered Successfully: projects/936666675765/locations/us-central1/evaluationRuns/7039953050324172800
Initial State: EvaluationRunState.PENDING

⏰ Polling execution status: [Current State: EvaluationRunState.PENDING] — sleeping 15s...
⏰ Polling execution status: [Current State: EvaluationRunState.RUNNING] — sleeping 15s...
⏰ Polling execution status: [Current State: EvaluationRunState.RUNNING] — sleeping 15s...
⏰ Polling execution status: [Current State: EvaluationRunState.RUNNING] — sleeping 15s...
⏰ Polling execution status: [Current State: EvaluationRunState.RUNNING] — sleeping 15s...
⏰ Polling execution status: [Current State: EvaluationRunState.RUNNING] — sleeping 15s...
⏰ Polling execution status: [Current State: EvaluationRunState.RUNNING] — sleeping 15s...

🏁 Lifecycle execution concluded! Terminal State: EvaluationRunState.SUCCEEDED


In [39]:
import json

if eval_run_handle.state == "SUCCEEDED" or eval_run_handle.state.name == "SUCCEEDED":
    # 1. Fetch deep model payload from Vertex AI
    completed_eval_data = client.evals.get_evaluation_run(
        name=eval_run_handle.name,
        include_evaluation_items=True
    )
    
    print("=" * 90)
    print(f"🥇 AGENTOPS EVALUATION PLATFORM RUN REPORT: {eval_run_handle.name.split('/')[-1]}")
    print("=" * 90)
    
    # 2. Extract strict dictionary from Pydantic object model schema
    try:
        eval_run_dict = completed_eval_data.model_dump()
    except Exception:
        try:
            eval_run_dict = completed_eval_data.dict()
        except Exception:
            eval_run_dict = getattr(completed_eval_data, "__dict__", {})

    item_results_block = eval_run_dict.get("evaluation_item_results", [])
    
    # Normalize varied array listings from the API payload trees
    if isinstance(item_results_block, dict):
        case_results = item_results_block.get("eval_case_results", [])
    else:
        case_results = item_results_block

    # ------------------------------------------------------------
    # STEP 1: PARSE INDIVIDUAL CASE ITEMS (QUESTION BREAKDOWN TABLE)
    # ------------------------------------------------------------
    print("\n📋 [UI SCREEN COMPONENT]: ATOMIC SAMPLE-ROW DATA BREAKDOWN TABLE")
    print("-" * 90)
    
    frontend_table_rows = []
    
    for idx, case_map in enumerate(case_results):
        if not isinstance(case_map, dict):
            continue
            
        case_index = case_map.get("eval_case_index", idx)
        candidates = case_map.get("response_candidate_results", [])
        if not candidates:
            continue
            
        primary_candidate = candidates[0]
        metric_results = primary_candidate.get("metric_results", {})
        
        parsed_row_data = {
            "index": case_index + 1,
            "user_prompt": f"Evaluation Test Case Matrix Row {case_index + 1}",
            "agent_response": "Rendered successfully within metrics detail logic path.",
            "scores_dictionary": {},
            "verdict_explanations": {}
        }
        
        # Traverse unique metrics targets
        for metric_key, metric_data in metric_results.items():
            if not isinstance(metric_data, dict):
                continue
                
            score_val = metric_data.get("score")
            parsed_row_data["scores_dictionary"][metric_key] = score_val
            
            # Map child rubrics verdicts lists blocks
            rubric_verdicts = metric_data.get("rubric_verdicts")
            verdict_summaries = []
            
            if isinstance(rubric_verdicts, list):
                for rubric in rubric_verdicts:
                    if not isinstance(rubric, dict):
                        continue
                    
                    evaluated_rubric = rubric.get("evaluated_rubric", {})
                    content_block = evaluated_rubric.get("content", {}) if isinstance(evaluated_rubric, dict) else {}
                    property_block = content_block.get("property", {}) if isinstance(content_block, dict) else {}
                    criteria_desc = property_block.get("description", "Criteria Assessment") if isinstance(property_block, dict) else "Criteria Assessment"
                    
                    verdict_outcome = rubric.get("verdict")
                    reasoning_text = rubric.get("reasoning", "No justification provided.")
                    
                    verdict_summaries.append({
                        "criteria": criteria_desc,
                        "passed": verdict_outcome,
                        "reasoning": reasoning_text
                    })
            
            parsed_row_data["verdict_explanations"][metric_key] = verdict_summaries
            
        frontend_table_rows.append(parsed_row_data)
        
        # Output interactive terminal visualizations for the notebook environment
        print(f"\n📍 [ROW #{parsed_row_data['index']}] {parsed_row_data['user_prompt']}")
        print(f" 🎛️ Metric Scores Summary: {json.dumps(parsed_row_data['scores_dictionary'])}")
        print(" 🧠 Judge Rubric Breakdown Analysis:")
        
        for metric_name, summaries in parsed_row_data["verdict_explanations"].items():
            print(f"    ▫️ Metric: {metric_name.upper()}")
            for item in summaries:
                status_icon = "✅ PASSED" if item["passed"] is True else ("❌ FAILED" if item["passed"] is False else "⚠️ NEUTRAL")
                print(f"       [{status_icon}] Check: {item['criteria']}")
                print(f"       └ Reasoning: {item['reasoning'].strip()}")
        print("." * 90)

    # ------------------------------------------------------------
    # STEP 2: GENERATE AGGREGATE SUMMARY METRICS (SCORECARD GAUGES)
    # ------------------------------------------------------------
    print("\n🔵 [UI SCREEN COMPONENT]: AGGREGATE SUMMARY STATS GAUGES")
    print("-" * 90)
    
    summary_metrics_payload = {}
    all_metric_keys = set()
    for row in frontend_table_rows:
        all_metric_keys.update(row["scores_dictionary"].keys())
        
    if all_metric_keys:
        for metric_key in all_metric_keys:
            scores_list = [
                row["scores_dictionary"][metric_key]
                for row in frontend_table_rows
                if row["scores_dictionary"].get(metric_key) is not None
            ]
            
            if scores_list:
                avg_score = sum(scores_list) / len(scores_list)
                min_score = min(scores_list)
                max_score = max(scores_list)
            else:
                avg_score, min_score, max_score = 0.0, 0.0, 0.0
                
            summary_metrics_payload[metric_key] = {
                "mean": round(avg_score, 2),
                "min": round(min_score, 2),
                "max": round(max_score, 2)
            }
            print(f" 🎯 Metric: {metric_key.upper():<30} | Mean Score: {summary_metrics_payload[metric_key]['mean']:.2f} | Min: {summary_metrics_payload[metric_key]['min']:.1f} | Max: {summary_metrics_payload[metric_key]['max']:.1f}")
    else:
        print("⚠️ No metrics values could be aggregated from evaluation cases.")

    # ------------------------------------------------------------
    # STEP 3: PREPARE FASTAPI PRODUCTION PAYLOAD REFERENCE OBJECT
    # ------------------------------------------------------------
    # This unified Python map matches the database requirements of your PRD
    unified_database_jsonb_mock = {
        "evaluation_run_id": eval_run_handle.name.split("/")[-1],
        "status": "complete",
        "aggregates": summary_metrics_payload,
        "items_breakdown_table": frontend_table_rows
    }
    print("\n✅ Database payload compilation complete. Metrics parsed cleanly from native arrays.")
    
else:
    print("⚠️ Active run tracking block still executing or failed to achieve execution success.")

🥇 AGENTOPS EVALUATION PLATFORM RUN REPORT: 7039953050324172800

📋 [UI SCREEN COMPONENT]: ATOMIC SAMPLE-ROW DATA BREAKDOWN TABLE
------------------------------------------------------------------------------------------

📍 [ROW #1] Evaluation Test Case Matrix Row 1
 🎛️ Metric Scores Summary: {"tool_use_quality_v1": 0.33333334, "instruction_following_v1": 1.0, "final_response_quality_v1": 1.0, "hallucination_v1": 1.0}
 🧠 Judge Rubric Breakdown Analysis:
    ▫️ Metric: TOOL_USE_QUALITY_V1
       [⚠️ NEUTRAL] Check: The agent's response acknowledges the user's request for dining recommendations.
       └ Reasoning: The agent's response is empty, therefore it does not acknowledge the user's request.
       [✅ PASSED] Check: The agent correctly refrains from making any tool calls, as no tools have been provided that can fulfill the user's request.
       └ Reasoning: The agent's response is empty and thus contains no tool calls, which is correct because no tools were provided.
       [⚠️ NEU

In [40]:
evaluation_item_results = eval_run_dict.get("evaluation_item_results", {})
case_results_preview = evaluation_item_results.get("eval_case_results", [])[:2]
print(f"Evaluation item results block sample preview: {json.dumps(case_results_preview, default=str, indent=2)}")

Evaluation item results block sample preview: [
  {
    "eval_case_index": 0,
    "response_candidate_results": [
      {
        "response_index": 0,
        "metric_results": {
          "tool_use_quality_v1": {
            "metric_name": "tool_use_quality_v1",
            "score": 0.33333334,
            "explanation": null,
            "rubric_verdicts": [
              {
                "evaluated_rubric": {
                  "rubric_id": null,
                  "content": {
                    "property": {
                      "description": "The agent's response acknowledges the user's request for dining recommendations."
                    }
                  },
                  "type": "INTENT:ACKNOWLEDGE_REQUEST",
                  "importance": "IMPORTANCE_UNSPECIFIED"
                },
                "verdict": null,
                "reasoning": "The agent's response is empty, therefore it does not acknowledge the user's request."
              },
              {
    